In [8]:
import pandas as pd
import re

# Ganti dengan path file Excel kamu
df = pd.read_csv("Balita Puskesmas.csv", header=None)
df.columns = ["JK", "Usia Saat Ukur", "Berat", "Tinggi", "BB/U", "ZS BB/U", "TB/U", "ZS TB/U", "BB/TB", "ZS BB/TB"]

# Konversi usia ke total bulan (desimal)
def parse_usia(s):
    m = re.match(r'(\d+) Tahun - (\d+) Bulan - (\d+) Hari', str(s))
    if m:
        tahun, bulan, hari = int(m.group(1)), int(m.group(2)), int(m.group(3))
        return round(tahun * 12 + bulan + hari / 30.44, 2)
    return None

df["Usia_Bulan"] = df["Usia Saat Ukur"].apply(parse_usia)

# Label encoding JK: L=0, P=1
df["JK"] = df["JK"].map({"L": 0, "P": 1})

# Hitung BMI
df["BMI"] = df["Berat"] / ((df["Tinggi"] / 100) ** 2)
df["BMI"] = df["BMI"].round(2)

# Buat kolom Status_Stunting dari ZS TB/U
def status_stunting(zs):
    try:
        zs = float(zs)
        if zs < -3:
            return "Sangat Pendek"
        elif zs < -2:
            return "Pendek"
        else:
            return "Normal"
    except:
        return None

df["Status_Stunting"] = df["ZS TB/U"].apply(status_stunting)

# Drop kolom yang tidak dipakai
df = df.drop(columns=["Usia Saat Ukur", "BB/U", "TB/U", "BB/TB", "ZS BB/U", "ZS BB/TB"])

# Reorder kolom sesuai ekspektasi training script
df = df[["JK", "Usia_Bulan", "Berat", "Tinggi", "BMI", "ZS TB/U", "Status_Stunting"]]

print(df.head())
print(f"\nShape: {df.shape}")
print(f"\nDistribusi Status_Stunting:")
print(df["Status_Stunting"].value_counts())

df.to_csv("dataset_clean.csv", index=False)
print("\nSaved to dataset_clean.csv")


   JK  Usia_Bulan  Berat  Tinggi    BMI  ZS TB/U Status_Stunting
0   0       13.49    9.6    76.0  16.62    -0.61          Normal
1   1       19.03    9.7    76.5  16.57    -1.76          Normal
2   0       14.03    8.7    74.5  15.67    -1.42          Normal
3   1       10.76    8.3    74.0  15.16     0.86          Normal
4   0       11.03    9.5    74.0  17.35     0.01          Normal

Shape: (1397, 7)

Distribusi Status_Stunting:
Status_Stunting
Normal           1324
Pendek             55
Sangat Pendek      18
Name: count, dtype: int64

Saved to dataset_clean.csv


In [9]:
"""
==============================================================
TRAINING RANDOM FOREST v2 - Deteksi Stunting Pada Balita
==============================================================
Input    : dataset_clean_v2.csv (hasil data_preparation_v2.py)
Output   : model_stunting_v2.pkl
Fitur    : JK, Usia_Bulan, Berat, Tinggi, BMI
Target   : Status_Stunting (Normal / Pendek / Sangat Pendek)

Perubahan dari v1:
  - Load dari dataset_clean_v2.csv (sudah undersample Normal=800)
  - Hapus fitur Rasio_BB_TB
  - Evaluasi utama: F1-macro & Recall per kelas (bukan akurasi)
  - Cross validation scoring: F1-macro
  - Tambah threshold analysis untuk kelas stunting
==============================================================
"""

import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, recall_score)

# ──────────────────────────────────────────────
# FASE 1: LOAD DATASET BERSIH v2
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 1: LOAD DATASET BERSIH v2")
print("=" * 55)

df = pd.read_csv('dataset_clean.csv')
print(f"✅ Dataset dimuat: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"   Kolom: {df.columns.tolist()}")

print("\nDistribusi kelas:")
for label, count in df['Status_Stunting'].value_counts().items():
    pct = count / len(df) * 100
    bar = 'X' * int(pct / 2)
    print(f"   {label:<15}: {count:>4} ({pct:.1f}%) {bar}")
print()


# ──────────────────────────────────────────────
# FASE 2: PISAH FITUR DAN TARGET
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 2: PISAH FITUR DAN TARGET")
print("=" * 55)

# [v2] Tanpa Rasio_BB_TB
feature_cols = ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'BMI']
X = df[feature_cols]
y_raw = df['Status_Stunting']

le = LabelEncoder()
y = le.fit_transform(y_raw)

print(f"✅ Fitur (X) : {feature_cols}")
print(f"✅ Target (y): {sorted(y_raw.unique().tolist())}")
print(f"   Encoded   : Normal=0 | Pendek=1 | Sangat Pendek=2")
print(f"   Shape X   : {X.shape}\n")


# ──────────────────────────────────────────────
# FASE 3: SPLIT TRAIN & TEST (80:20)
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 3: SPLIT TRAIN & TEST (80:20 Stratified)")
print("=" * 55)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Data training : {X_train.shape[0]} baris")
print(f"✅ Data testing  : {X_test.shape[0]} baris")

# Cek distribusi kelas di test set
print("\nDistribusi kelas di test set:")
for kode, label in enumerate(le.classes_):
    count = (y_test == kode).sum()
    print(f"   {label:<15}: {count} baris")
print()


# ──────────────────────────────────────────────
# FASE 4: TRAINING MODEL RANDOM FOREST
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 4: TRAINING MODEL RANDOM FOREST")
print("=" * 55)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
print(f"✅ Model berhasil ditraining!")
print(f"   Jumlah pohon : {rf_model.n_estimators}")
print(f"   Class weight : balanced\n")


# ──────────────────────────────────────────────
# FASE 5: EVALUASI MODEL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 5: EVALUASI MODEL")
print("=" * 55)

y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)

# --- Metrik Utama ---
acc        = accuracy_score(y_test, y_pred)
f1_macro   = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

# Recall per kelas
recall_per_kelas = recall_score(y_test, y_pred, average=None)

print(f"📊 METRIK UTAMA:")
print(f"   Akurasi          : {acc * 100:.2f}%")
print(f"   F1-Score Macro   : {f1_macro * 100:.2f}%  ← metrik utama")
print(f"   F1-Score Weighted: {f1_weighted * 100:.2f}%")
print()

print(f"📊 RECALL PER KELAS (seberapa banyak kasus tertangkap):")
for label, recall in zip(le.classes_, recall_per_kelas):
    status = "✅" if recall >= 0.80 else "⚠️ "
    bar = 'X' * int(recall * 20)
    print(f"   {status} {label:<15}: {recall*100:.1f}% {bar}")
print()

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm,
    index=[f'Aktual: {c}' for c in le.classes_],
    columns=[f'Prediksi: {c}' for c in le.classes_]
)
print(cm_df.to_string())
print()

# --- Analisis Kesalahan ---
print("\n📊 ANALISIS KESALAHAN PREDIKSI STUNTING:")
idx_pendek = list(le.classes_).index('Pendek')
idx_sangat = list(le.classes_).index('Sangat Pendek')
idx_normal = list(le.classes_).index('Normal')

miss_pendek_ke_normal = cm[idx_pendek][idx_normal]
miss_sangat_ke_normal = cm[idx_sangat][idx_normal]
miss_sangat_ke_pendek = cm[idx_sangat][idx_pendek]

print(f"   Pendek salah prediksi ke Normal       : {miss_pendek_ke_normal} kasus")
print(f"   Sangat Pendek salah prediksi ke Normal: {miss_sangat_ke_normal} kasus")
print(f"   Sangat Pendek salah prediksi ke Pendek: {miss_sangat_ke_pendek} kasus")
print()


# ──────────────────────────────────────────────
# FASE 6: CROSS VALIDATION (5-Fold) — F1 Macro
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 6: CROSS VALIDATION (5-Fold, F1-Macro)")
print("=" * 55)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Scoring F1-macro (metrik utama)
cv_f1 = cross_val_score(rf_model, X, y, cv=cv, scoring='f1_macro')
# Accuracy tetap ditampilkan sebagai pembanding
cv_acc = cross_val_score(rf_model, X, y, cv=cv, scoring='accuracy')

print(f"✅ CV F1-Macro tiap fold  : {[f'{s*100:.2f}%' for s in cv_f1]}")
print(f"   Rata-rata F1-Macro    : {cv_f1.mean()*100:.2f}%  ← acuan utama")
print(f"   Std Deviasi F1-Macro  : {cv_f1.std()*100:.2f}%")
print()
print(f"✅ CV Akurasi tiap fold   : {[f'{s*100:.2f}%' for s in cv_acc]}")
print(f"   Rata-rata Akurasi     : {cv_acc.mean()*100:.2f}%")
print(f"   Std Deviasi Akurasi   : {cv_acc.std()*100:.2f}%")
print()


# ──────────────────────────────────────────────
# FASE 7: FEATURE IMPORTANCE
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 7: FEATURE IMPORTANCE")
print("=" * 55)

importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False)

print("Kontribusi tiap fitur terhadap model:")
for feat, imp in feat_imp.items():
    bar = 'X' * int(imp * 50)
    print(f"   {feat:<12}: {imp:.4f} ({imp*100:.1f}%) {bar}")
print()


# ──────────────────────────────────────────────
# FASE 8: THRESHOLD ANALYSIS
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 8: THRESHOLD ANALYSIS")
print("=" * 55)
print("Cek apakah menurunkan threshold stunting meningkatkan recall...\n")

thresholds = [0.5, 0.40, 0.35, 0.30]

for thr in thresholds:
    y_pred_thr = []
    for proba in y_proba:
        prob_normal = proba[idx_normal]
        prob_pendek = proba[idx_pendek]
        prob_sangat = proba[idx_sangat]

        # Jika prob stunting (pendek/sangat pendek) >= threshold → prediksi stunting
        if prob_sangat >= thr:
            y_pred_thr.append(idx_sangat)
        elif prob_pendek >= thr:
            y_pred_thr.append(idx_pendek)
        else:
            y_pred_thr.append(idx_normal)

    y_pred_thr = np.array(y_pred_thr)
    f1_thr  = f1_score(y_test, y_pred_thr, average='macro')
    acc_thr = accuracy_score(y_test, y_pred_thr)
    rec_thr = recall_score(y_test, y_pred_thr, average=None)

    print(f"   Threshold={thr} → Akurasi={acc_thr*100:.1f}% | F1-Macro={f1_thr*100:.1f}%")
    for label, rec in zip(le.classes_, rec_thr):
        print(f"      Recall {label:<15}: {rec*100:.1f}%")
    print()


# ──────────────────────────────────────────────
# FASE 9: SIMPAN MODEL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 9: SIMPAN MODEL")
print("=" * 55)

with open("balita_puskesmas.pkl", "wb") as f:
    pickle.dump(rf_model, f)

with open("balita_puskesmas.pkl", "wb") as f:
    pickle.dump(le, f)

print("✅ File tersimpan:")
print("   📦 model_stunting_v2.pkl          → Model Random Forest v2")
print("   📦 label_encoder_stunting_v2.pkl  → Label encoder\n")


# ──────────────────────────────────────────────
# FASE 10: UJI PREDIKSI MANUAL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 10: UJI PREDIKSI MANUAL")
print("=" * 55)

# [v2] Tanpa Rasio_BB_TB
berat  = 9.0
tinggi = 80.0

contoh_data = pd.DataFrame([{
    'JK'        : 0,   # P=0, L=1
    'Usia_Bulan': 24,
    'Berat'     : berat,
    'Tinggi'    : tinggi,
    'BMI'       : berat / ((tinggi / 100) ** 2),
}])

hasil_encoded = rf_model.predict(contoh_data)[0]
hasil_label   = le.inverse_transform([hasil_encoded])[0]
probabilitas  = rf_model.predict_proba(contoh_data)[0]

print("Input contoh balita:")
print(f"   JK=P, Usia=24 bln, BB={berat}kg, TB={tinggi}cm")
print(f"   BMI={contoh_data['BMI'].values[0]:.2f}")
print(f"\nHasil Prediksi : {hasil_label}")
print(f"Probabilitas   :")
for kelas, prob in zip(le.classes_, probabilitas):
    status = " ← prediksi" if kelas == hasil_label else ""
    print(f"   {kelas:<15}: {prob*100:.1f}%{status}")

FASE 1: LOAD DATASET BERSIH v2
✅ Dataset dimuat: 1397 baris, 7 kolom
   Kolom: ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'BMI', 'ZS TB/U', 'Status_Stunting']

Distribusi kelas:
   Normal         : 1324 (94.8%) XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
   Pendek         :   55 (3.9%) X
   Sangat Pendek  :   18 (1.3%) 

FASE 2: PISAH FITUR DAN TARGET
✅ Fitur (X) : ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'BMI']
✅ Target (y): ['Normal', 'Pendek', 'Sangat Pendek']
   Encoded   : Normal=0 | Pendek=1 | Sangat Pendek=2
   Shape X   : (1397, 5)

FASE 3: SPLIT TRAIN & TEST (80:20 Stratified)
✅ Data training : 1117 baris
✅ Data testing  : 280 baris

Distribusi kelas di test set:
   Normal         : 265 baris
   Pendek         : 11 baris
   Sangat Pendek  : 4 baris

FASE 4: TRAINING MODEL RANDOM FOREST
✅ Model berhasil ditraining!
   Jumlah pohon : 200
   Class weight : balanced

FASE 5: EVALUASI MODEL
📊 METRIK UTAMA:
   Akurasi          : 94.64%
   F1-Score Macro   : 62.47%  ← metrik utama
  